## Llama.cpp / Ollama benutzen

### 1. Vorbereitung (Terminal)

**Installation:**

* **Mac/Linux:** `curl -fsSL https://ollama.com/install.sh | sh`
* **Windows:** Installer von [ollama.com](https://ollama.com) benutzen.

**Modell laden (Terminal):**
Wir nutzen `qwen2.5:0.5b` (nur ~300MB, läuft schnell genug auf Laptop-CPU).

```bash
ollama pull qwen2.5:0.5b

```

**Erster Test (CLI):**
Interaktiver Chat im Terminal.

```bash
ollama run qwen2.5:0.5b

```
Tippe:

`Wie kann man mit Deep Learning Hunde von Katzen unterscheiden (also, warum funktioniert das)? Zähle nur 5-10 Stichpunkte auf.`

Lesen und amüsieren (0.5 Milliarden Parameter sind eben nicht 50 Milliarden),
ggf. auch noch Vergleich mit

`how can deep learning distinguish cats from dogs / how does it actually work and is not magic? List 5-10 key ingredients in bullet points.`

dann mit /bye beenden.

**Server Check:**
Ollama läuft standardmäßig auf Port 11434.
Browser öffnen: `http://localhost:11434` (Sollte "Ollama is running" zeigen).
Unter `http://127.0.0.1:11434/v1/models` sollte Qwen 2.5 (0.5B) aufgeführt sein.

---

### 2. Python Setup

```
pip install ollama llama-cpp-python

```

---

### 3. Semantische Vektoren (Embeddings)

Ideal für Klassifizierung oder Suche.
Wandelt Text in einen festen Vektor (Größe je nach Modell, z.B. 896 Dimensionen) um.

In [1]:
import ollama
import numpy as np

TEXT = "Heute ist es kalt draußen."

response = ollama.embeddings(model='qwen2.5:0.5b', prompt=TEXT)

vector = np.array(response["embedding"])

print(f"Input: '{TEXT}'")
print(f"Vektor Dimensionen: {vector.shape}")
print(f"Erste 5 Werte: {vector[:5]}")
print(f"Größter und kleinster Eintrag: {np.min(vector)}, {np.max(vector)}")
print(f"Länge des Vektors: {np.linalg.norm(vector)}")

# TODO: Diesen 'vector' weiterverarbeiten...

Input: 'Heute ist es kalt draußen.'
Vektor Dimensionen: (896,)
Erste 5 Werte: [ 2.8132689  -1.64779329  7.00515461 -0.9352715  -3.80784774]
Größter und kleinster Eintrag: -179.5752716064453, 112.2237777709961
Länge des Vektors: 297.6088469441567


---

### 4. Wahrscheinlichkeitsverteilungen (Logits)

Statt Text zu generieren, schauen wir uns die Wahrscheinlichkeiten für das **nächste Token** an. Dazu benutzen wir jetzt nicht Ollama, sondern direkt llama.cpp (über den Python Wrapper).

Das Modell im GGUF Format ohne Ollama zu benutzen, gibt es direkt bei HuggingFace:
https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF
Wir benutzen die Q4_k_m-Variante:
https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct-GGUF/resolve/main/qwen2.5-0.5b-instruct-q4_k_m.gguf

In [2]:
from llama_cpp import Llama
import numpy as np

TEXT = "The color of the sky is"

MODEL_PATH = "./qwen2.5-0.5b-instruct-q4_k_m.gguf" 

llm = Llama(model_path=MODEL_PATH,
            logits_all=True,
            verbose=False,
            n_ctx=512)

def get_next_token_probs(text):
    llm.reset()
    tokens = llm.tokenize(text.encode("utf-8"))
    llm.eval(tokens)
    
    # Logits des letzten Tokens holen
    logits = np.array(llm.scores[len(tokens)-1])
    # Achtung: llm.scores[-1] wäre falsch, denn:
    assert len(llm.scores) == 512
    assert len(tokens) != 512 # we only do short input here :-)
    
    # Softmax, um Wahrscheinlichkeiten zu erhalten
    probs = np.exp(logits) / np.sum(np.exp(logits))
    return probs

# Test
probs = get_next_token_probs(TEXT)
print(probs)

print(f"Vokabular-Größe: N={probs.shape[0]}, Wahrscheinlichkeit eines Token bei Gleichverteilung wäre 1/N = {(1 / probs.shape[0]):.7f}")
best_id = np.argmax(probs)
print(f"Vorhersage nächstes Token: {llm.detokenize([best_id])} mit id {best_id} und Wahrscheinlichkeit {probs[best_id]:.4f}")

llama_context: n_ctx_per_seq (512) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


[9.2745000e-07 7.5749966e-07 5.0188063e-08 ... 7.0229017e-11 7.0295325e-11
 7.0229017e-11]
Vokabular-Größe: N=151936, Wahrscheinlichkeit eines Token bei Gleichverteilung wäre 1/N = 0.0000066
Vorhersage nächstes Token: b' changing' mit id 10018 und Wahrscheinlichkeit 0.1211


Wir schauen uns die Verteilung der wahrscheinlichsten Token etwas genauer an:

In [3]:
k = 25

top_k_ids = np.argsort(probs)[-k:][::-1]

print(f"Top {k} completions for '{TEXT}'")

for token_id in top_k_ids:
    probability = probs[token_id]
    token_text = "'" + llm.detokenize([token_id]).decode("utf-8") + "'"
    print(f"{token_text: <20}\t(ID: {token_id})\t Wahrscheinlichkeit: {(probability*100):.2f}%")

Top 25 completions for 'The color of the sky is'
' changing'         	(ID: 10018)	 Wahrscheinlichkeit: 12.11%
' always'           	(ID: 2677)	 Wahrscheinlichkeit: 6.45%
' blue'             	(ID: 6303)	 Wahrscheinlichkeit: 3.86%
' determined'       	(ID: 10838)	 Wahrscheinlichkeit: 3.04%
' very'             	(ID: 1602)	 Wahrscheinlichkeit: 2.97%
' mainly'           	(ID: 14576)	 Wahrscheinlichkeit: 2.97%
' a'                	(ID: 264)	 Wahrscheinlichkeit: 2.94%
' dark'             	(ID: 6319)	 Wahrscheinlichkeit: 2.83%
' influenced'       	(ID: 27061)	 Wahrscheinlichkeit: 2.70%
' usually'          	(ID: 5990)	 Wahrscheinlichkeit: 2.64%
' caused'           	(ID: 8881)	 Wahrscheinlichkeit: 2.50%
' __'               	(ID: 1304)	 Wahrscheinlichkeit: 2.49%
' ______'           	(ID: 32671)	 Wahrscheinlichkeit: 2.49%
' affected'         	(ID: 11495)	 Wahrscheinlichkeit: 2.25%
' the'              	(ID: 279)	 Wahrscheinlichkeit: 2.25%
' often'            	(ID: 3545)	 Wahrscheinlichkeit: 2.19%
' 

Wir schauen uns die Wahrscheinlichkeit für bestimmte Token an:

In [4]:
red_id = llm.tokenize(" red".encode("utf-8"))[0]
red_probability = probs[red_id]
print(f"' red'  \t(ID: {red_id}) \t | Wahrscheinlichkeit: {(red_probability*100):.2f}%")

' red'  	(ID: 2518) 	 | Wahrscheinlichkeit: 0.74%


### 5. Strukturierter Output (JSON)

Wir zwingen das LLM, JSON auszugeben, und nutzen das Python-Paket Pydantic, um zu garantieren, dass die Datentypen (Integer, String, etc.) stimmen.

In [7]:
from pydantic import BaseModel
import ollama

# 1. Wir definieren unser Schema
class Person(BaseModel):
    name: str
    age: int
    occupation: str
    location: str

our_keys = list(Person.model_fields.keys())

input_text = "Yesterday I met Sabine. She is 29 years old, works as a doctor and loves her life in Munich."
print(f" Raw input: '{input_text}'")
print(f" We want to extract: '{our_keys}'")

# 2. Anfrage an Qwen via Ollama mit format='json'
response = ollama.chat(
    model='qwen2.5:0.5b',
    messages=[{
        'role': 'user', 
        'content': f"Extract information about the person from this text into JSON using '{our_keys}: '{input_text}'"
    }],
    format='json'
)

raw_json = response['message']['content']
print(f"Roh-JSON vom LLM: {raw_json}")

# 3. Validierung & Umwandlung in Python-Objekt
person = Person.model_validate_json(raw_json)

print(f"Python Objekt: {person}")
print(f"Zugriff auf Eigenschaft: {person.name}")
assert type(person.age) == int

 Raw input: 'Yesterday I met Sabine. She is 29 years old, works as a doctor and loves her life in Munich.'
 We want to extract: '['name', 'age', 'occupation', 'location']'
Roh-JSON vom LLM: {
  "name": "Sabine",
  "age": 29,
  "occupation": "doctor",
  "location": "Munich"
}
Python Objekt: name='Sabine' age=29 occupation='doctor' location='Munich'
Zugriff auf Eigenschaft: Sabine
